# 03 -- Structural Alignment Quality Investigation

Characterises the RMSD distribution from the stratified alignment sample
(300 clusters drawn across size bands, ~17K pairwise alignments onto the
highest-resolution reference in each cluster).

**Goal:** decide an RMSD threshold for water pooling.
Entries failing the threshold are excluded from their cluster.

**Sample:** `alignment_sample_results.json` -- 50 clusters per size band:

| Size band | Clusters sampled |
|-----------|------------------|
| 4-6       | 50               |
| 7-12      | 50               |
| 13-25     | 50               |
| 26-50     | 50               |
| 51-200    | 50               |
| 201+      | all (22)         |

Alignments: gemmi C++ BLOSUM62 sequence alignment + Kabsch superposition on matched Ca atoms.

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sys.path.insert(0, '../scripts')
from cluster_filter import CONSERVATION_DIR

with open(CONSERVATION_DIR / 'alignment_sample_results.json') as f:
    sample = json.load(f)

print(f'Clusters in sample: {len(sample)}')

all_rmsds, all_n_aligned, total_failed = [], [], 0
for cluster in sample:
    total_failed += cluster['n_failed']
    for pair in cluster['pairs']:
        all_rmsds.append(pair['rmsd'])
        all_n_aligned.append(pair['n_aligned'])

rmsds     = np.array(all_rmsds)
n_aligned = np.array(all_n_aligned)
print(f'Total pairwise alignments:  {len(rmsds):,}')
print(f'Failed alignments:          {total_failed}')

## 1. RMSD percentile table

In [ ]:
print('RMSD distribution across 17,213 pairwise alignments')
print(f'  Min:    {rmsds.min():.3f} Ang')
print(f'  Median: {np.median(rmsds):.2f} Ang')
print(f'  Mean:   {rmsds.mean():.2f} Ang')
print(f'  Max:    {rmsds.max():.2f} Ang')
print()
print('Percentiles:')
for p in [50, 75, 84, 90, 95, 99, 99.5]:
    print(f'  p{p:<5.1f}  {np.percentile(rmsds, p):.2f} Ang')
print()
print('Fraction of alignments below threshold:')
for t in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    print(f'  < {t:.1f} Ang:  {(rmsds < t).mean()*100:.1f}%')

## 2. RMSD distribution -- histogram

In [ ]:
img = mpimg.imread(CONSERVATION_DIR / 'alignment_rmsd_distribution.png')
fig, ax = plt.subplots(figsize=(13, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Interpretation and suggested threshold

The distribution is **strongly bimodal**:

- **~95% of alignments fall below 3.0 Ang** (mostly < 1 Ang -- rigid same-protein
  structures differing only by crystal packing).
- **~5% are outliers (3-26 Ang RMSD)** -- likely apo vs. holo, open vs. closed,
  or shifted disordered loops.

The 95th-99th percentile jump (2.54 Ang to 18.19 Ang) is a clean gap with no
population between ~3-18 Ang.

**Suggested threshold: 3.0 Ang RMSD.**

Entries above this threshold will be excluded from water pooling for their cluster.
Clusters where >= 25% of entries exceed it may warrant structural sub-clustering
(future work; 25/272 sampled clusters fall here).

---

Please visually inspect the representative pairs in section 4 (PyMOL / ChimeraX)
to confirm the threshold.

## 4. Representative pairs for visual inspection

In [ ]:
target_rmsds = [0.50, 1.50, 3.00, 6.00, 20.00]
labels = [
    'Excellent (well below any threshold)',
    'Good (within 3 Ang threshold)',
    'Borderline (~95th percentile, at threshold)',
    'Outlier (above threshold, likely conformational change)',
    'Extreme outlier (near-random superposition)',
]

reps = []
for target in target_rmsds:
    best, best_diff = None, float('inf')
    for cluster in sample:
        ref = cluster['reference_pdb']
        for pair in cluster['pairs']:
            diff = abs(pair['rmsd'] - target)
            if diff < best_diff:
                best_diff = diff
                best = (pair['pdb_id'], ref, pair['rmsd'], pair['n_aligned'])
    reps.append(best)

print(f'{"Target":>8}  {"Actual":>8}  {"Query":>6}  {"Ref":>6}  {"N_aln":>6}  Label')
print('-' * 85)
for target, label, (qry, ref, rmsd, na) in zip(target_rmsds, labels, reps):
    print(f'{target:>8.2f}  {rmsd:>8.2f}  {qry:>6}  {ref:>6}  {na:>6}  {label}')

print()
print('PyMOL:   fetch <query>; fetch <ref>; align <query>, <ref>')
print('ChimeraX: open <query>; open <ref>; matchmaker #1 to #2')

## 5. Failure rate by cluster size

Checking whether large clusters have higher alignment failure rates.

In [ ]:
size_bands = [(4, 6), (7, 12), (13, 25), (26, 50), (51, 200), (201, 10000)]
labels_sz  = ['4-6', '7-12', '13-25', '26-50', '51-200', '201+']

print(f'{"Band":>7}  {"Clusters":>8}  {"Pairs":>8}  {"Failed":>7}  {"Fail%":>6}  {"Median":>7}  {"<3Ang%":>7}')
print('-' * 65)
for (lo, hi), lbl in zip(size_bands, labels_sz):
    band = [c for c in sample if lo <= c['cluster_size'] <= hi]
    if not band:
        continue
    r = np.array([p['rmsd'] for c in band for p in c['pairs']])
    nf = sum(c['n_failed'] for c in band)
    total = len(r) + nf
    print(f'{lbl:>7}  {len(band):>8}  {len(r):>8}  {nf:>7}  {nf/total*100:>5.1f}%  {np.median(r):>7.2f}  {(r<3).mean()*100:>6.1f}%')

## 6. High-RMSD fraction per cluster

How many clusters have a substantial fraction of high-RMSD entries?
Informs drop-per-entry vs. split-cluster strategy.

In [ ]:
THRESHOLD = 3.0

high_fracs = np.array([
    np.mean([p['rmsd'] >= THRESHOLD for p in c['pairs']])
    for c in sample if c['pairs']
])

print(f'Fraction of cluster entries >= {THRESHOLD} Ang RMSD:')
print(f'  0% high-RMSD:     {(high_fracs == 0).sum():>4} clusters ({(high_fracs == 0).mean()*100:.0f}%)')
print(f'  <10% high-RMSD:   {(high_fracs < 0.10).sum():>4} clusters ({(high_fracs < 0.10).mean()*100:.0f}%)')
print(f'  <25% high-RMSD:   {(high_fracs < 0.25).sum():>4} clusters ({(high_fracs < 0.25).mean()*100:.0f}%)')
print(f'  >=25% high-RMSD:  {(high_fracs >= 0.25).sum():>4} clusters ({(high_fracs >= 0.25).mean()*100:.0f}%)')
print(f'  >=50% high-RMSD:  {(high_fracs >= 0.50).sum():>4} clusters ({(high_fracs >= 0.50).mean()*100:.0f}%)')
print()
print('Most clusters have a small outlier tail -- per-entry exclusion is sufficient.')
print('The >=25% clusters may warrant sub-clustering in a future step.')